### Explore A Jain's IEI pipeline
### Julian Moran
### 2026-02-09

In [65]:
import os

from dotenv import load_dotenv
from pathlib import Path

import polars as pl
import numpy as np

# Env
pl.Config.set_tbl_rows(10)
load_dotenv("../.env")
INSTALL_PATH = os.environ["INSTALL_PATH"]

# Pipeline IO

Codebase at `` 

1. DefenseFinder RefSeq IDs --> UniProt IDs
    - DefenseFinder: db of all bacterial genes implicated in antiphage systems
    - RefSeq bacterial accession IDs:   denote a bacterial immunoprotein; often begin with `WP`
    - UniProt access IDs:               denote a protein; e.g. `A0A7P0TAE1_HUMAN`

<br>

2. UniProt IDs --> Foldseek cluster ID

<br>

3. Foldseek cluster ID --> related human cluster ID

<br>

4. UniProt IDs --> UniprotKB annotations

<br>


# Questions

1. How does `01_RefSeq2Uniprot_mapping/` fit into the pipeline?
    - Are data in `01_RefSeq2Uniprot_mapping/` the output from pipeline step 2.?
    - Are data in `griid_gene_list/`, `gasdermin/`, `vipirin/`, `zorya/` prioritized subsets from `01_RefSeq2Uniprot_mapping/`?
    - If so, why are the subsets substantially larger than the original data?
    - `WP_063110969.1`, from `gasdermin/2025-07-26_gasdermin_annotated.xlsx`, is not in `data_local/01_RefSeq2Uniprot_mapping/2025-06-24_RefSeq2Uniprot_mapping.xlsx`

<br>

2. What is the relationship between each UniProt accession number and its corresponding cluster?
    - i.e. in datasets like `zorya/2025-07-27_zorya_annotated.tsv`?


<br>

3. Why do we see identical rows in the annotated data?
    - best to denote rows as `RefSeq_ID`,`UniProt_ID`,`FoldSeq_cluster`,`Similar_human_gene` 
    - e.g. in `zorya/2025-07-27_zorya_annotated.tsv`, see multiple rows for `WP_063192462.1`,`A0A142CZW3`,`A0A142CZW3`,`B4DLD8`


<br>

# Filtering criteria

1. Select for human proteins in same cluster as bacterial
    - `defense_system_cluster_id == cluster_id`
    - in annotated data, `defense_system_cluster_id` denotes the FoldSeek cluster that contains the query DefenseFinder protein
    - in annotated data, `cluster_id` denotes the most-similar target FoldSeek cluster containing a human protein 


2. Select for human proteins closest in AA length to each bacterial protein


2. Select for closest human genes that are in EAGLE definitive



<br>

# Missing read permissions

1. `00_run_pipeline.sh`
2. `scripts/05_generate_slurm_file.py`
3. `scripts/06_find_human_foldseek_matches.py`
4. `scripts/07_run_foldseek.py`
5. `scripts/07_submit_run_foldseek.sh`
6. `scripts/08_add_annotations-optimized-new_metrics.py`

In [4]:
# ============================================================
#                             Args
# ============================================================

args = {
    "data_file_foldseek_uniprot": f"{INSTALL_PATH}/data_local/02_FoldSeek_Results_w_Uniprot_matches/2025-07-26_batch_0_annotated.tsv",
    "data_file_GRIID": f"{INSTALL_PATH}/data_local/griid_gene_list/2025-07-30_data_filtered_for_griid_genes.tsv",
    "data_file_gasdermin": f"{INSTALL_PATH}/data_sync/gasdermin/2025-07-26_gasdermin_annotated.tsv",
    "data_file_vipirin": f"{INSTALL_PATH}/data_sync/viperin/2025-07-26_viperin_annotated.tsv",
    "data_file_zorya": f"{INSTALL_PATH}/data_local/zorya/2025-07-27_zorya_annotated.tsv"
}

In [90]:
# ============================================================
#                             In
# ============================================================

data_ann = {}
for key, path in args.items():
    if "data_file_" in key and not "foldseek" in key:
        data_key = key.replace("data_file_", "")
        data_ann[data_key] = pl.read_csv(
            args[key],
            has_header=True,
            separator="\t",
            schema_overrides={
                "duplicate_count": pl.Int64,
                "rank": pl.Int64,
                "evalue": pl.Float64,
                "cluFlag": pl.Int64,
                "fident": pl.Float64,
                "alnlen": pl.Int64,
                "mismatch": pl.Int64,
                "gapopen": pl.Float64,
                "qstart": pl.Int64,
                "qend": pl.Int64,
                "tstart": pl.Int64,
                "tend": pl.Int64,
                "evalue_foldseek": pl.Float64,
                "bits": pl.Int64,
                "Query_length": pl.Int64,
                "Human_prot_total_length": pl.Int64,
                "Human_domain_percentage": pl.Float64,
                "Query_percentage": pl.Float64,
                "Query_overlap_w_Human_protein": pl.Float64,
                "Target_length": pl.Int64,
                "Bacterial_prot_total_length": pl.Int64,
                "Bacterial_domain_percentage": pl.Float64,
                "Target_percentage": pl.Float64,
                "Target_overlap_w_Bacterial_protein": pl.Float64,
                "Bacterial_Human_Len_Ratio": pl.Float64,
                "Query_Target_Overlap_Length": pl.Float64,
                "Overlap_ratio": pl.Float64,
            },
            infer_schema_length=10000
        )

df_foldseek_uniprot = pl.read_csv(
    args["data_file_foldseek_uniprot"],
    separator="\t",
    has_header=True
)

df_foldseek_uniprot[0:19]

accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,i64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,i64,f64
"""WP_001223208.1""","""MazEF""","""MazEF""","""Shigella dysenteriae,Escherich…","""GCF_003017995_NZ_CP027368_MazE…","""GCF_002156845.1_NZ_CP021339_05…","""GCF_019428585.1_NZ_CP080119_01…",671,6,"""A0A024L8H8""","""S3FMY0""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""nan..nan""",null,null,null,null,null,null,null,null,"""nan..nan""",null,83,null,null,null,null,null,null,null,null,null
"""WP_003414296.1""","""Cas""","""CAS_Class1-Subtype-III-A""","""Mycobacterium tuberculosis,Myc…","""GCF_014884645_NZ_CP043996_CAS_…","""GCF_002975475.1_NZ_CP027035_02…","""GCF_013010385.1_NZ_CP053092_02…",264,23,"""A0A045ICU9""","""A0A1I6JKI6""",null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,"""nan..nan""",null,null,null,null,null,null,null,null,"""nan..nan""",null,124,null,null,null,null,null,null,null,null,null
"""WP_002885155.1""","""AbiE""","""AbiE""","""Klebsiella sp. P1927,Klebsiell…","""GCF_022354545_NZ_CP055087_AbiE…","""GCF_022494255.1_NZ_CP058743_04…","""GCF_003286975.1_NZ_CP030172_04…",454,12,"""A0A060VEP4""","""J1I8N4""","""A0A6J2X4D6""","""Q5TF85""",0.02082,1,0.098,297,214,0,179,475,10,247,0.001215,42,"""unreviewed""","""Q5TF85_HUMAN""","""polynucleotide adenylyltransfe…","""TENT5A FAM46A hCG_401094""","""Homo sapiens (Human)""",null,"""hCG_401094""","""TENT5A""","""FAM46A""","""UP000005640: Chromosome 6""",null,null,"""poly(A) RNA polymerase activit…","""poly(A) RNA polymerase activit…","""GO:1990817""",null,null,null,"""IPR012937; TET5.;""","""PTHR12974; PRION-LIKE- Q/N-RIC…","""PF07984; NTP_transf_7; 1.;""",null,"""SM01153; DUF1693; 1.;""",null,"""179.0..475.0""",297,523,null,null,null,null,null,null,"""10.0..247.0""",238,305,null,null,null,null,null,null,0.583174,238,0.780328
"""WP_002885155.1""","""AbiE""","""AbiE""","""Klebsiella sp. P1927,Klebsiell…","""GCF_022354545_NZ_CP055087_AbiE…","""GCF_022494255.1_NZ_CP058743_04…","""GCF_003286975.1_NZ_CP030172_04…",454,12,"""A0A060VEP4""","""J1I8N4""","""G1SF99""","""Q5VWP2""",0.08138,1,0.094,295,265,0,49,343,10,302,0.0009699,39,"""reviewed""","""TET5C_HUMAN""","""Terminal nucleotidyltransferas…","""TENT5C FAM46C""","""Homo sapiens (Human)""",null,null,"""TENT5C""","""FAM46C""","""UP000005640: Chromosome 1""","""in utero embryonic development…","""centrosome [GO:0005813]; cytop…","""centrosome [GO:0005813]; cytop…","""poly(A) RNA polymerase activit…","""GO:0001701; GO:0003723; GO

In [91]:
# ============================================================
#                Clean, stack annotated data
# ============================================================

def remove_dup_rows(df: pl.DataFrame) -> int:
    df_dup_counted = (
        df.group_by(df.columns)
        .len()
    )
    dups_count = (
        df_dup_counted.filter(pl.col("len") > 1)
        .select((pl.col("len") - 1).sum())
        .item()
    )
    print(f"Number of dup rows: {dups_count:,}")
    return df_dup_counted.drop("len"), dups_count


data_ann_cl = {}
for data_key, df in data_ann.items():
    data_ann_cl[data_key], dups_count = remove_dup_rows(df)

    # Write out cleaned version
    if dups_count > 0:
        key = [k for k in args if data_key in k][0]
        out_path = Path(args[key].replace(".tsv", "_cleaned.tsv"))
        data_ann_cl[data_key].write_csv(
            out_path,
            separator="\t",
            include_header=True
        )

df_ann_cl = pl.concat(
    [df for df in data_ann_cl.values()],
    how="diagonal"
)

print(f"Number of rows after cleaning, v-stacking: {len(df_ann_cl):,}")
df_ann_cl

Number of dup rows: 0
Number of dup rows: 0
Number of dup rows: 266
Number of dup rows: 40,578
Number of rows after cleaning, v-stacking: 321,575


accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio,GRIID_gene
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64,str
"""WP_001077932.1""","""PsyrTA""","""PsyrTA""","""Escherichia coli""","""GCF_017088465_NZ_CP070914_Psyr…","""GCF_011331065.1_NZ_CP049081_04…","""GCF_002012245.1_NZ_CP019000_03…",14,189,"""A0A0A3ANF5""","""A0A2T0Q2J3""","""A0A4W5N4R5""","""A0A2R8YDT5""",2.8320e-11,1,0.177,368,296,0.0,5,372,9,369,3.2390e-16,452,"""unreviewed""","""A0A2R8YDT5_HUMAN""","""RNA helicase (EC 3.6.4.13)""","""DDX3X""","""Homo sapiens (Human)""",null,null,"""DDX3X""",null,"""UP000005640: Chromosome X""",null,"""cytosol [GO:0005829]""","""cytosol [GO:0005829]; ATP bind…","""ATP binding [GO:0005524]; hydr…","""GO:0003676; GO:0003724; GO:000…","""cd18051; DEADc_DDX3; 1.;""cd187…","""3.40.50.300:FF:000160; ATP-dep…","""3.40.50.300; P-loop containing…","""IPR011545; DEAD/DEAH_box_helic…","""PTHR47958; ATP-DEPENDENT RNA H…","""PF00270; DEAD; 1.;""PF00271; He…","""PS00039; DEAD_ATP_HELICASE; 1.…","""SM00487; DEXDc; 1.;""SM00490; H…","""NP_001350748.1; NM_001363819.1…","""5.0..372.0""",368,475,"""228..389""",89.51,39.4,77.47,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…","""9.0..369.0""",361,710,"""230..397""",83.33,38.78,50.85,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",1.494737,361.0,0.76,"""Yes"""
"""WP_003159898.1""","""Rst_HelicaseDUF2290""","""Rst_HelicaseDUF2290""","""Pseudomonas aeruginosa""","""GCF_013305815_NZ_CP050053_Rst_…","""GCF_013305815.1_NZ_CP050053_01…","""GCF_003813025.1_NZ_CP033835_02…",22,181,"""Q9I3X2""","""A0A380WVS4""","""A0A084VX50""","""A0A0A0MT12""",0.02702,1,0.123,213,152,0.0,55,228,171,383,0.00214,34,"""unreviewed""","""A0A0A0MT12_HUMAN""","""RNA helicase (EC 3.6.4.13)""","""DDX39B""","""Homo sapiens (Human)""",null,null,"""DDX39B""",null,"""UP000005640: Chromosome 6""",null,"""nuclear speck [GO:0016607]""","""nuclear speck [GO:0016607]; AT…","""ATP binding [GO:0005524]; hydr…","""GO:0003676; GO:0003724; GO:000…","""cd17950; DEADc_DDX39; 1.;""","""3.40.50.300:FF:000168; DEAD-bo…","""3.40.50.300; P-loop containing…","""IPR011545; DEAD/DEAH_box_helic…","""PTHR47959:SF1; ATP-DEPENDENT R…","""PF00270; DEAD; 1.;""","""PS51192; HELICASE_ATP_BIND_1; …","""SM00487; DEXDc; 1.;""",null,"""55.0..228.0""",174,235,"""76..235""",95.62,87.93,74.04,"""['Helicase ATP-binding']""","""['ECO:0000259|PROSITE:PS51192'…","""171.0..383.0""",213,711,null,null,null,null,null,null,3.025532,174.0,0.740426,"""No"""
"""WP_014757523.1""","""Cas""","""CAS_Class1-Subtype-I-B""","""Thermoanaerobacterium aotearoe…","""GCF_000307585_NC_017992_CAS_Cl…","""GCF_000307585.2_NC_017992_0057…","""GCF_000307585.2_NC_017992_0058…",2,201,"""W9EJ04""","""A0A3A9WFL8""","""E4YUU0""","""J3KT12""",0.001837,1,0

In [94]:
# ============================================================
#                     Filter annotated data
# ============================================================

def filter_data_ann(df: pl.DataFrame) -> pl.DataFrame:

    df = df.filter(
        (pl.col("defense_system_cluster_id") == pl.col("cluster_id")) &
        (pl.col("Organism") == "Homo sapiens (Human)")
    )
    print(f"N rows after filtering: {len(df)}")

    return df

filter_data_ann(df_ann_cl)


N rows after filtering: 904


accession_in_sys,type,subtype,species,sys_id,sys_beg,sys_end,duplicate_count,rank,defense_system_protein_id,defense_system_cluster_id,cluster_id,protein_id,evalue,cluFlag,fident,alnlen,mismatch,gapopen,qstart,qend,tstart,tend,evalue_foldseek,bits,Reviewed,Entry Name,Protein names,Gene Names,Organism,Gene Names (ordered locus),Gene Names (ORF),Gene Names (primary),Gene Names (synonym),Proteomes,Gene Ontology (biological process),Gene Ontology (cellular component),Gene Ontology (GO),Gene Ontology (molecular function),Gene Ontology IDs,CDD,FunFam,Gene3D,InterPro,PANTHER,Pfam,PROSITE,SMART,RefSeq,Query_range,Query_length,Human_prot_total_length,Human_prot_domain_match,Human_domain_percentage,Query_percentage,Query_overlap_w_Human_protein,Human_domain_note,Human_domain_evidence,Target_range,Target_length,Bacterial_prot_total_length,Bacterial_prot_domain_match,Bacterial_domain_percentage,Target_percentage,Target_overlap_w_Bacterial_protein,Bacterial_domain_note,Bacterial_domain_evidence,Bacterial_Human_Len_Ratio,Query_Target_Overlap_Length,Overlap_ratio,GRIID_gene
str,str,str,str,str,str,str,i64,i64,str,str,str,str,f64,i64,f64,i64,i64,f64,i64,i64,i64,i64,f64,i64,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,str,i64,i64,str,f64,f64,f64,str,str,str,i64,i64,str,f64,f64,f64,str,str,f64,f64,f64,str
"""WP_128105540.1""","""Zorya""","""Zorya_TypeI""","""Acetobacter oryzoeni""","""GCF_004014775_NZ_CP042808_Zory…","""GCF_004014775.2_NZ_CP042808_01…","""GCF_004014775.2_NZ_CP042808_01…",1,202,"""A0A5B9GHV7""","""A0A355UP19""","""A0A355UP19""","""Q9H4L7""",0.0,1,0.196,657,524,0.0,370,1026,362,1014,9.7780e-28,762,"""reviewed""","""SMRCD_HUMAN""","""SWI/SNF-related matrix-associa…","""SMARCAD1 KIAA1122""","""Homo sapiens (Human)""",null,null,"""SMARCAD1""","""KIAA1122""","""UP000005640: Chromosome 4""","""chromatin remodeling [GO:00063…","""chromatin [GO:0000785]; hetero…","""chromatin [GO:0000785]; hetero…","""ATP binding [GO:0005524]; ATP …","""GO:0000018; GO:0000729; GO:000…","""cd17998; DEXHc_SMARCAD1; 1.;""c…","""3.40.50.10810:FF:000014; SWI/S…","""3.40.50.300; P-loop containing…","""IPR003892; CUE.;""IPR014001; He…","""PTHR10799; SNF2/RAD54 HELICASE…","""PF00271; Helicase_C; 1.;""PF001…","""PS51140; CUE; 2.;""PS51192; HEL…","""SM00487; DEXDc; 1.;""SM00490; H…","""NP_001121901.1; NM_001128429.3…","""370.0..1026.0""",657,1026,"""509..677""",100.0,25.72,64.04,"""['Helicase ATP-binding']""","""['ECO:0000255|PROSITE-ProRule:…","""362.0..1014.0""",653,1028,"""505..697""",100.0,29.56,63.52,"""['Helicase ATP-binding']""","""['ECO:0000259|PROSITE:PS51192'…",1.001949,653.0,0.636452,"""No"""
"""WP_142817048.1""","""Zorya""","""Zorya_TypeI""","""Rhodoferax sediminis""","""GCF_006970865_NZ_CP035503_Zory…","""GCF_006970865.1_NZ_CP035503_00…","""GCF_006970865.1_NZ_CP035503_00…",1,202,"""A0A515D6D4""","""A0A7S4UXI5""","""A0A7S4UXI5""","""B3KX98""",0.0,2,0.219,177,129,0.0,708,873,917,1093,0.000001,228,"""unreviewed""","""B3KX98_HUMAN""","""cDNA FLJ45012 fis, clone BRAWH…",null,"""Homo sapiens (Human)""",null,null,null,null,null,null,"""nucleus [GO:0005634]""","""nucleus [GO:0005634]; ATP bind…","""ATP binding [GO:0005524]; heli…","""GO:0004386; GO:0005524; GO:000…","""cd16569; RING-HC_SHPRH-like; 1…","""3.30.40.10:FF:000162; E3 ubiqu…","""3.40.50.300; P-loop containing…","""IPR052583; ATP-helicase/E3_Ub-…","""PTHR45865:SF1; E3 UBIQUITIN-PR…","""PF00271; Helicase_C; 1.;""PF213…","""PS51194; HELICASE_CTER; 1.;""PS…","""SM00490; HELICc; 1.;""SM00184; …","""NP_001036148.2; NM_001042683.2…","""708.0..873.0""",166,882,"""713..871""",100.0,95.78,18.82,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…","""917.0..1093.0""",177,1094,"""923..1086""",100.0,92.66,16.18,"""['Helicase C-terminal']""","""['ECO:0000259|PROSITE:PS51194'…",1.240363,166.0,0.188209,"""No"""
"""WP_013954635.1""","""RM""","""RM_Type_II""","""Mycoplasmopsis bovis""","""GCF_014854615_NZ_CP062195_RM_T…","""GCF_021497085.1_NZ_CP040774_0